# Kafka Streaming Sandbox
Ejercicios breves para explorar el clúster local. Ejecuta este notebook dentro del servicio `KafkaLab`.

In [ ]:
import json
from lab.kafka import wait_for_cluster, ensure_topic, topic_rows, watermarks
from lab.config import producer, consumer
from lab.show import table

cluster = wait_for_cluster()
print('Brokers:', sorted(cluster.brokers))

In [ ]:
topic = 'notebook-sandbox'
ensure_topic(topic, partitions=3, replication=3, reset=True)
table(topic_rows(topic))

In [ ]:
writer = producer(**{'acks': 'all', 'enable.idempotence': True})
for sequence in range(30):
    key = f'M{sequence % 3:05d}'
    value = json.dumps({'sequence': sequence, 'kwh': sequence / 10})
    writer.produce(topic, key=key, value=value)
assert writer.flush(30) == 0
table([{'partition': p, 'low': bounds[0], 'high': bounds[1]}
       for p, bounds in watermarks(topic).items()])

In [ ]:
reader = consumer('notebook-sandbox-reader', **{'enable.auto.commit': False})
reader.subscribe([topic])
records = []
while len(records) < 30:
    message = reader.poll(5)
    if message is not None and message.error() is None:
        records.append(json.loads(message.value()))
reader.close()
print('records:', len(records), 'first:', records[:3])

## Siguiente
Continúa con `lessons/01_kraft_cluster.py` y sigue el orden del README. Los topics del sandbox pueden borrarse y recrearse con `reset=True`.